# Devoir 2 : Système RAG pour le Code de la Route Marocain 🚗
## Retrieval-Augmented Generation sur textes juridiques arabes

**Approche** : tous les LLMs sont appelés via l'**API HuggingFace Inference** — aucun téléchargement local de modèle requis.

---
### Architecture
```
Question → [FAISS Retriever] → Chunks pertinents → [HF Inference API LLM] → Réponse
```

### 3 LLMs comparés
| Modèle | Famille | Appel |
|--------|---------|-------|
| **Qwen/Qwen3-8B** | Alibaba Qwen3 | HF API |
| **mistralai/Mistral-7B-Instruct-v0.3** | Mistral AI | HF API |
| **microsoft/Phi-3.5-mini-instruct** | Microsoft (GPT-like) | HF API |


## 1. Installation des dépendances


In [1]:
# Dependencies are listed in ../requirements.txt — install with:
#     pip install -r ../requirements.txt
print('Dependencies should be installed via requirements.txt')


Dependencies should be installed via requirements.txt


## 2. Imports


In [2]:
import os, re, warnings
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from huggingface_hub import InferenceClient
import gradio as gr
from IPython.display import display

warnings.filterwarnings('ignore')
print("✅ Imports OK")


✅ Imports OK


## 3. Configuration — Token HF et paramètres


In [ ]:
# ── Token HuggingFace ──────────────────────────────────────
# Remplacez la valeur ci-dessous par votre token (hf_...)
# ou définissez la variable d'environnement HF_TOKEN avant de lancer le notebook
HF_TOKEN = os.environ.get("HF_TOKEN", "YOUR_TOKEN_HERE")

if HF_TOKEN == "VOTRE_TOKEN_ICI" or not HF_TOKEN:
    raise ValueError(
        "❌ Token HuggingFace manquant !\n"
        "  Définissez la variable d'environnement HF_TOKEN :\n"
        "    export HF_TOKEN=hf_xxxxxxxxxxxxxxxx   (dans le terminal)\n"
        "  ou remplacez 'VOTRE_TOKEN_ICI' ci-dessus par votre token."
    )
print(f"✅ Token HF chargé ({HF_TOKEN[:8]}...)")

# ── Chemins CSV ────────────────────────────────────────────
CSV_CANDIDATES = [
    "/kaggle/input/devoir1-export/export_final.csv",
    "/kaggle/input/devoir1/export_final.csv",
    str(Path(__file__).parent.parent / "devoir1" / "export_final.csv")
        if "__file__" in dir() else "",
    str(Path.home() / "NLP" / "devoir1" / "export_final.csv"),
    "../devoir1/export_final.csv",
    "export_final.csv",
]

# ── Paramètres RAG ─────────────────────────────────────────
RETRIEVAL_K     = 5      # chunks récupérés par requête
OOD_THRESHOLD   = 0.28   # seuil cosinus pour détection OOD
CHUNK_MAX_WORDS = 120
CHUNK_OVERLAP   = 25

# ── Modèles LLM (via HF Inference API) ────────────────────
MODEL_CONFIGS = {
    "Qwen3-8B": {
        "model_id": "Qwen/Qwen3-8B",
        "family":   "Alibaba Qwen3",
    },
    "Mistral-7B": {
        "model_id": "mistralai/Mistral-7B-Instruct-v0.3",
        "family":   "Mistral AI",
    },
    "Phi-3.5-mini": {
        "model_id": "microsoft/Phi-3.5-mini-instruct",
        "family":   "Microsoft (GPT-like)",
    },
}

print("\n📋 Modèles configurés (HF Inference API) :")
for name, cfg in MODEL_CONFIGS.items():
    print(f"  ✅ {name} — {cfg['family']} ({cfg['model_id']})")


✅ Token HF chargé (hf_Imbcm...)

📋 Modèles configurés (HF Inference API) :
  ✅ Qwen3-8B — Alibaba Qwen3 (Qwen/Qwen3-8B)
  ✅ Mistral-7B — Mistral AI (mistralai/Mistral-7B-Instruct-v0.3)
  ✅ Phi-3.5-mini — Microsoft (GPT-like) (microsoft/Phi-3.5-mini-instruct)


---
## Section 1 : Préparation des données

Chargement du CSV produit dans le Devoir 1 (126 articles du القانون 52.05),
nettoyage du texte arabe, puis découpage en chunks.


In [5]:
# ── 1.1 Chargement du CSV ──────────────────────────────────
csv_path = None
for p in CSV_CANDIDATES:
    if p and Path(p).exists():
        csv_path = p
        break

if csv_path is None:
    raise FileNotFoundError(
        "❌ export_final.csv introuvable.\n"
        "  Placez-le dans ../devoir1/export_final.csv ou définissez le chemin."
    )

df = pd.read_csv(csv_path)
print(f"✅ CSV chargé : {len(df)} articles | {len(df.columns)} colonnes")
print(f"   Source : {csv_path}")
display(df[['article_id', 'type_infraction', 'niveau_gravite', 'amende_fixe', 'sanction_presente']].head(5))


✅ CSV chargé : 126 articles | 17 colonnes
   Source : /Users/beethoven/NLP/devoir1/export_final.csv


,article_id,type_infraction,niveau_gravite,amende_fixe,sanction_presente
0,page_1,general,information,NaN,False
1,page_2,general,information,NaN,False
2,page_3,general,information,NaN,False
3,page_4,general,information,NaN,False
4,page_5,general,information,NaN,False


In [6]:
# ── 1.2 Nettoyage et normalisation du texte arabe ──────────
def normalize_arabic(text) -> str:
    if pd.isna(text) or str(text).strip() in ('', 'nan'):
        return ""
    text = str(text).strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def build_article_text(row) -> str:
    parts = []
    art_id = str(row.get('article_id', '')).strip()
    if art_id and art_id != 'nan':
        parts.append(f"المادة {art_id}:")
    infraction = normalize_arabic(row.get('infraction_desc', ''))
    if infraction:
        parts.append(infraction)
    sanction = normalize_arabic(row.get('sanction_desc', ''))
    if sanction:
        parts.append(f"العقوبة: {sanction}")
    amende = row.get('amende_fixe', '')
    if pd.notna(amende) and str(amende) not in ('nan', ''):
        parts.append(f"الغرامة: {amende} درهم")
    return ' '.join(parts)

df['full_text'] = df.apply(build_article_text, axis=1)
df = df[df['full_text'].str.len() > 30].reset_index(drop=True)
print(f"✅ {len(df)} articles après nettoyage")
print(f"\nExemple :\n{df['full_text'].iloc[2][:300]}")


✅ 126 articles après nettoyage

Exemple :
المادة page_3: هموكحلل هماعلا هناملاا 3 هدالما اهيلا راشلما هدلما مارصنا دعب ،جراخلاب هملسم هقايس هصخر يلع نيلصاحلا نيقياسلا يلع بجي ليدبت اوبلطي نا وا ،هيبرغلما هقايسلا هصخر يلع لوصحلا تاناحتملا اومدقتي نا ،هقباسلا هدالما يف .هيلاولما تارقفلل اقيبطت هقايسلل مهتصخر قافتا برغلماب اهطبري هلود لبق نم ه


In [7]:
# ── 1.3 Découpage en chunks ────────────────────────────────
def chunk_text(text: str, max_words: int = CHUNK_MAX_WORDS, overlap: int = CHUNK_OVERLAP) -> list:
    words = text.split()
    if len(words) <= max_words:
        return [text]
    chunks, start = [], 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunks.append(' '.join(words[start:end]))
        if end == len(words):
            break
        start += max_words - overlap
    return chunks

texts: list    = []
metadata: list = []

for _, row in df.iterrows():
    for chunk in chunk_text(row['full_text']):
        texts.append(chunk)
        metadata.append({
            'article_id':      str(row.get('article_id', '')),
            'type_infraction': str(row.get('type_infraction', '')),
            'niveau_gravite':  str(row.get('niveau_gravite', '')),
            'amende_fixe':     str(row.get('amende_fixe', '')),
            'localisation':    str(row.get('localisation', '')),
        })

wc = [len(t.split()) for t in texts]
print(f"✅ {len(df)} articles → {len(texts)} chunks")
print(f"   Taille moy : {np.mean(wc):.0f} mots | Max : {max(wc)} mots")


✅ 126 articles → 444 chunks
   Taille moy : 107 mots | Max : 120 mots


---
## Section 2 : Indexation et Recherche (Retriever)

- **Embedding** : `intfloat/multilingual-e5-small` — supporte l'arabe, léger (118M params)
- **Index** : FAISS `IndexFlatIP` — recherche cosinus exacte


In [8]:
# ── 2.1 Modèle d'embedding ────────────────────────────────
print("⏳ Chargement de multilingual-e5-small...")
embedding_model = SentenceTransformer("intfloat/multilingual-e5-small")
dim = embedding_model.get_sentence_embedding_dimension()
print(f"✅ Embedding prêt — {dim} dimensions")


⏳ Chargement de multilingual-e5-small...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

... (3 outputs left out)

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

✅ Embedding prêt — 384 dimensions


In [9]:
# ── 2.2 Encodage + index FAISS ────────────────────────────
print(f"⏳ Encodage de {len(texts)} chunks...")
passages   = ["passage: " + t for t in texts]
embeddings = embedding_model.encode(
    passages, batch_size=64, show_progress_bar=True, normalize_embeddings=True
)
embeddings = np.array(embeddings, dtype=np.float32)

index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"\n✅ Index FAISS : {index.ntotal} vecteurs × {dim} dimensions")


⏳ Encodage de 444 chunks...


Batches:   0%|          | 0/7 [00:00<?, ?it/s]


✅ Index FAISS : 444 vecteurs × 384 dimensions


In [10]:
# ── 2.3 Fonction de recherche ─────────────────────────────
def retrieve(query: str, k: int = RETRIEVAL_K, min_score: float = 0.0) -> list:
    q_vec = embedding_model.encode(
        ["query: " + query], normalize_embeddings=True
    ).astype(np.float32)
    scores, indices = index.search(q_vec, k)
    return [
        {'text': texts[i], 'score': float(s), 'meta': metadata[i]}
        for s, i in zip(scores[0], indices[0]) if s >= min_score
    ]

# Test
print("🔍 Test retrieval — 'ما هي شروط الحصول على رخصة السياقة؟'")
for r in retrieve("ما هي شروط الحصول على رخصة السياقة؟"):
    print(f"  {r['score']:.3f} | Art.{r['meta']['article_id']} | {r['text'][:70]}...")


🔍 Test retrieval — 'ما هي شروط الحصول على رخصة السياقة؟'
  0.791 | Art.page_33 | تاناويح قياس وا هبكرم قياس وا لجار لك : هيمومعلا قيرطلا لمعتسم ؛ هيموم...
  0.787 | Art.page_40 | نم ، هقايسلا هصخر يلع لوصحلا اهتقايس بلطتت ،هبكرم هقايس 4 11 .هقايسلا ...
  0.786 | Art.page_117 | ؛ لمعلا اهب يراجلا هيميظنتلاو يا يلع وا يقرطلا ريوشتلل هحول يلع رخا لك...
  0.786 | Art.page_36 | جاجزلا يلع ،اهتلومح وا هتبكرم داعبا ببسب ،رارضا قاحلا نود ريسلا هيناكم...
  0.785 | Art.page_37 | زواجتت لا هدلم هقايسلا هصخر فيقوتب ارارق هرادلاا ردصت : صاخشلال يعامجل...


---
## Section 3 : Appel LLM via HF Inference API

Fonction générique qui appelle n'importe quel modèle HuggingFace via `InferenceClient.chat_completion`.
Aucun téléchargement local — la génération se fait sur les serveurs HuggingFace.


In [11]:
# ── Appel LLM via HF Inference API ────────────────────────
def call_llm(model_key: str, messages: list) -> str:
    cfg    = MODEL_CONFIGS[model_key]
    client = InferenceClient(model=cfg["model_id"], token=HF_TOKEN)
    try:
        resp = client.chat_completion(messages=messages, max_tokens=400)
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[Erreur API {model_key}: {e}]"

# Smoke test rapide (1 appel)
print("🧪 Test API avec Mistral-7B...")
_test = call_llm("Mistral-7B", [{"role": "user", "content": "Réponds en un mot : bonjour"}])
print(f"   Réponse : {_test}")
print("✅ API HF opérationnelle")


🧪 Test API avec Mistral-7B...
   Réponse : [Erreur API Mistral-7B: (Request ID: Root=1-6a03590d-31f8b160415dac496096d72d;b2240b77-9194-43fd-94f6-a7136ca63ad3)

Bad request:
{'message': "The requested model 'mistralai/Mistral-7B-Instruct-v0.3' is not a chat model.", 'type': 'invalid_request_error', 'param': 'model', 'code': 'model_not_supported'}]
✅ API HF opérationnelle


---
## Section 4 : Pipeline RAG complet

```
Question
  → is_out_of_domain() ?  → OOD : message d'avertissement
  → retrieve()            → top-K chunks FAISS
  → build_messages()      → prompt avec contexte
  → call_llm()            → réponse via HF API
  → résultat + références
```


In [12]:
# ── 4.1 Détection OOD ─────────────────────────────────────
def is_out_of_domain(query: str) -> tuple:
    results   = retrieve(query, k=3, min_score=0.0)
    max_score = max((r['score'] for r in results), default=0.0)
    return max_score < OOD_THRESHOLD, max_score

# ── 4.2 Construction du prompt ─────────────────────────────
def build_messages(query: str, docs: list) -> list:
    context = "\n\n".join(
        f"[المصدر {i} — المادة {d['meta']['article_id']}]\n{d['text']}"
        for i, d in enumerate(docs, 1)
    )
    system = (
        "أنت مساعد قانوني متخصص في قانون السير على الطرق المغربي (القانون 52.05). "
        "استخدم فقط المعلومات الواردة في السياق أدناه للإجابة. "
        "إذا لم تجد الإجابة، قل ذلك بوضوح. "
        "اذكر دائماً رقم المادة المصدر."
    )
    user = f"السياق:\n{context}\n\nالسؤال: {query}"
    return [
        {"role": "system", "content": system},
        {"role": "user",   "content": user},
    ]

# ── 4.3 Fonction principale RAG ────────────────────────────
def rag_answer(query: str, model_key: str) -> dict:
    ood, max_score = is_out_of_domain(query)
    if ood:
        return {
            'answer':    "⚠️ هذا السؤال خارج نطاق قانون السير على الطرق.",
            'docs':      [],
            'ood':       True,
            'max_score': max_score,
        }
    docs     = retrieve(query, k=RETRIEVAL_K)
    messages = build_messages(query, docs)
    answer   = call_llm(model_key, messages)
    return {'answer': answer, 'docs': docs, 'ood': False, 'max_score': max_score}

print("✅ Pipeline RAG défini")


✅ Pipeline RAG défini


In [13]:
# ── Test rapide pipeline complet ──────────────────────────
_q = "ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟"
print(f"❓ {_q}\n")
_res = rag_answer(_q, "Qwen3-8B")
print(f"📝 Réponse Qwen3-8B :\n{_res['answer']}")
print(f"\n📚 {len(_res['docs'])} docs récupérés | Score max : {_res['max_score']:.3f}")


❓ ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟

📝 Réponse Qwen3-8B :
<think>
Okay, the user is asking about the minimum age to obtain a driver's license in Morocco according to Law 52.05. Let me check the provided sources.

Looking at the sources, there are several mentions of the law, but I need to find the specific article that states the age requirement. 

In Source 3, which is page 2, it mentions "المادة page_2" and talks about the law 52.05. It says "هموكحلل هماعلا هناملاا 52.05 مقر نوناقلا 1.10.07 مقر فيرشلا ريهظلا هذيفنتب رداصلا قرطلا يلع ريسلا هنودمب قلعتلما..." This seems to be about the legal framework but doesn't specify the age. 

Then, in Source 5, page 92, there's a mention of "هلاعا 58و 38 نيتدالما يف هددحلما لاجلاا هلكش هرادلاا ددحت..." which might refer to articles 58 and 38. However, the exact age isn't here. 

Looking at Source 2, page 62, it discusses penalties and fines but not the age. Source 4, page 61, mentions "هدحاو هنسو رهشا )6( هتس نيب 60" which could be a

---
## Section 5 : Comparaison des 3 LLMs

Mêmes questions, même retriever → seule la génération LLM varie.


In [14]:
TEST_QUESTIONS = [
    "ما هي شروط الحصول على رخصة السياقة في المغرب؟",
    "ما هي العقوبة المقررة لقيادة السيارة بدون رخصة؟",
    "كم عدد النقاط المخصصة لرخصة السياقة؟",
    "ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟",
    "ما هي إجراءات تجديد رخصة القيادة بعد انتهاء صلاحيتها؟",
]

comparison_results = {}

for model_key in MODEL_CONFIGS:
    print(f"\n{'='*60}")
    print(f"🤖  {model_key} — {MODEL_CONFIGS[model_key]['family']}")
    print('='*60)
    rows = []
    for i, q in enumerate(TEST_QUESTIONS, 1):
        print(f"  [{i}/{len(TEST_QUESTIONS)}] {q[:55]}...")
        res = rag_answer(q, model_key)
        short = res['answer'][:200] + ("..." if len(res['answer']) > 200 else "")
        print(f"  → {short}\n")
        rows.append({
            'Question':  q,
            'Réponse':   res['answer'],
            'Docs':      len(res['docs']),
            'Score max': round(res['max_score'], 3),
        })
    comparison_results[model_key] = rows

print("\n✅ Comparaison des 3 LLMs terminée !")



🤖  Qwen3-8B — Alibaba Qwen3
  [1/5] ما هي شروط الحصول على رخصة السياقة في المغرب؟...
  → <think>
Okay, the user is asking about the conditions to obtain a driver's license in Morocco. Let me check the provided sources. 

Looking at the sources, they mention various articles related to tra...

  [2/5] ما هي العقوبة المقررة لقيادة السيارة بدون رخصة؟...
  → <think>
Okay, the user is asking about the penalty for driving without a license in Moroccan law, specifically under Law 52.05. Let me check the provided sources.

Looking through the sources, they al...

  [3/5] كم عدد النقاط المخصصة لرخصة السياقة؟...
  → <think>
Okay, the user is asking about the number of points allocated for a driver's license in Moroccan traffic law, specifically under Law 52.05. Let me check the provided sources to find the answer...

  [4/5] ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟...
  → <think>
Okay, let's tackle this question about the minimum age for obtaining a driver's license in Morocco according t

In [15]:
# ── Synthèse comparative ───────────────────────────────────
summary = []
for model_key, rows in comparison_results.items():
    summary.append({
        'Modèle':                      model_key,
        'Famille':                     MODEL_CONFIGS[model_key]['family'],
        'Score récup. moy.':           round(float(np.mean([r['Score max'] for r in rows])), 3),
        'Longueur rép. moy. (mots)':   int(np.mean([len(r['Réponse'].split()) for r in rows])),
    })
print("📊 SYNTHÈSE COMPARATIVE\n")
display(pd.DataFrame(summary))


📊 SYNTHÈSE COMPARATIVE



,Modèle,Famille,Score récup. moy.,Longueur rép. moy. (mots)
0,Qwen3-8B,Alibaba Qwen3,0.784,263
1,Mistral-7B,Mistral AI,0.784,24
2,Phi-3.5-mini,Microsoft (GPT-like),0.784,28


---
## Section 6 : Évaluation des performances du Retriever

- **Precision@5** = docs pertinents récupérés / 5
- **Recall@5** = docs pertinents récupérés / total docs pertinents
- **F1** = moyenne harmonique


In [16]:
GROUND_TRUTH = {
    "ما هي شروط الحصول على رخصة السياقة في المغرب؟":    ["10", "11", "12"],
    "كم عدد النقاط المخصصة لرخصة السياقة؟":              ["22", "23", "27"],
    "ما هي مدة صلاحية حامل رخصة السياقة؟":               ["37", "38"],
    "ما هي التزامات السائق المهني في المغرب؟":            ["40", "41", "42"],
    "ما هي إجراءات استرجاع النقاط المفقودة من الرخصة؟":  ["33", "34", "35"],
    "كيف يتم سحب رخصة السياقة؟":                         ["31", "32"],
    "ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟":    ["11"],
}

def evaluate_retriever(ground_truth: dict, k: int = RETRIEVAL_K) -> pd.DataFrame:
    rows = []
    for query, relevant_ids in ground_truth.items():
        results       = retrieve(query, k=k)
        retrieved_ids = set(r['meta']['article_id'] for r in results)
        relevant_set  = set(relevant_ids)
        tp            = len(relevant_set & retrieved_ids)
        precision     = tp / len(retrieved_ids) if retrieved_ids else 0.0
        recall        = tp / len(relevant_set)  if relevant_set  else 0.0
        f1 = (2*precision*recall / (precision+recall)) if (precision+recall) > 0 else 0.0
        rows.append({
            'Question':       (query[:55]+'…') if len(query)>55 else query,
            f'Precision@{k}': round(precision, 3),
            f'Recall@{k}':    round(recall, 3),
            'F1':              round(f1, 3),
            'Attendus':        relevant_ids,
            'Récupérés':       sorted(retrieved_ids),
        })
    df_ev = pd.DataFrame(rows)
    mean_row = pd.DataFrame([{
        'Question':       '📊 MOYENNE',
        f'Precision@{k}': round(df_ev[f'Precision@{k}'].mean(), 3),
        f'Recall@{k}':    round(df_ev[f'Recall@{k}'].mean(), 3),
        'F1':              round(df_ev['F1'].mean(), 3),
        'Attendus': [], 'Récupérés': [],
    }])
    return pd.concat([df_ev, mean_row], ignore_index=True)

eval_df = evaluate_retriever(GROUND_TRUTH)
print(f"📊 Évaluation Retriever (k={RETRIEVAL_K})\n")
display(eval_df[['Question', f'Precision@{RETRIEVAL_K}', f'Recall@{RETRIEVAL_K}', 'F1']])
m = eval_df.iloc[-1]
print(f"\nPrecision@{RETRIEVAL_K}: {m[f'Precision@{RETRIEVAL_K}']} | Recall@{RETRIEVAL_K}: {m[f'Recall@{RETRIEVAL_K}']} | F1: {m['F1']}")


📊 Évaluation Retriever (k=5)



,Question,Precision@5,Recall@5,F1
0,ما هي شروط الحصول على رخصة السياقة في المغرب؟,0.0,0.0,0.0
1,كم عدد النقاط المخصصة لرخصة السياقة؟,0.0,0.0,0.0
2,ما هي مدة صلاحية حامل رخصة السياقة؟,0.0,0.0,0.0
3,ما هي التزامات السائق المهني في المغرب؟,0.0,0.0,0.0
4,ما هي إجراءات استرجاع النقاط المفقودة من الرخصة؟,0.0,0.0,0.0
5,كيف يتم سحب رخصة السياقة؟,0.0,0.0,0.0
6,ما هو الحد الأدنى لسن الحصول على رخصة السياقة؟,0.0,0.0,0.0
7,📊 MOYENNE,0.0,0.0,0.0



Precision@5: 0.0 | Recall@5: 0.0 | F1: 0.0


---
## Section 7 : Détection des questions hors domaine (OOD)

Si le score cosinus maximal entre la question et le corpus est **< 0.28** → question hors domaine.


In [17]:
OOD_CASES = [
    ("ما هي شروط الحصول على رخصة السياقة؟",    False),
    ("ما عقوبة السياقة في حالة سكر؟",           False),
    ("ما هي أنواع رخص القيادة في المغرب؟",      False),
    ("كيف يتم خصم النقاط من رخصة السياقة؟",    False),
    ("ما هي التزامات السائق المهني؟",            False),
    ("ما هي عاصمة فرنسا؟",                     True),
    ("كيف أصنع كعكة الشوكولاتة؟",              True),
    ("من هو مخترع الهاتف؟",                    True),
    ("What is machine learning?",               True),
    ("ما هو الناتج الداخلي الخام للمغرب؟",     True),
]

print(f"🔍 Test OOD (seuil = {OOD_THRESHOLD})\n")
ood_rows, correct = [], 0
for query, expected_ood in OOD_CASES:
    detected_ood, score = is_out_of_domain(query)
    ok = detected_ood == expected_ood
    if ok: correct += 1
    label = "🔴 OOD" if detected_ood else "🟢 In-domain"
    icon  = "✅" if ok else "❌"
    print(f"{icon} Score={score:.3f} | {label} (attendu: {'OOD' if expected_ood else 'In-domain'}) | {query[:55]}")
    ood_rows.append({'Question': query, 'OOD attendu': expected_ood,
                     'OOD détecté': detected_ood, 'Score': round(score,3), 'Correct': ok})
print(f"\n📊 Précision OOD : {correct}/{len(OOD_CASES)} = {correct/len(OOD_CASES):.1%}")
display(pd.DataFrame(ood_rows))


🔍 Test OOD (seuil = 0.28)

✅ Score=0.791 | 🟢 In-domain (attendu: In-domain) | ما هي شروط الحصول على رخصة السياقة؟
✅ Score=0.796 | 🟢 In-domain (attendu: In-domain) | ما عقوبة السياقة في حالة سكر؟
✅ Score=0.767 | 🟢 In-domain (attendu: In-domain) | ما هي أنواع رخص القيادة في المغرب؟
✅ Score=0.796 | 🟢 In-domain (attendu: In-domain) | كيف يتم خصم النقاط من رخصة السياقة؟
✅ Score=0.796 | 🟢 In-domain (attendu: In-domain) | ما هي التزامات السائق المهني؟
❌ Score=0.728 | 🟢 In-domain (attendu: OOD) | ما هي عاصمة فرنسا؟
❌ Score=0.788 | 🟢 In-domain (attendu: OOD) | كيف أصنع كعكة الشوكولاتة؟
❌ Score=0.760 | 🟢 In-domain (attendu: OOD) | من هو مخترع الهاتف؟
❌ Score=0.736 | 🟢 In-domain (attendu: OOD) | What is machine learning?
❌ Score=0.761 | 🟢 In-domain (attendu: OOD) | ما هو الناتج الداخلي الخام للمغرب؟

📊 Précision OOD : 5/10 = 50.0%


,Question,OOD attendu,OOD détecté,Score,Correct
0,ما هي شروط الحصول على رخصة السياقة؟,False,False,0.791,True
1,ما عقوبة السياقة في حالة سكر؟,False,False,0.796,True
2,ما هي أنواع رخص القيادة في المغرب؟,False,False,0.767,True
3,كيف يتم خصم النقاط من رخصة السياقة؟,False,False,0.796,True
4,ما هي التزامات السائق المهني؟,False,False,0.796,True
5,ما هي عاصمة فرنسا؟,True,False,0.728,False
6,كيف أصنع كعكة الشوكولاتة؟,True,False,0.788,False
7,من هو مخترع الهاتف؟,True,False,0.760,False
8,What is machine learning?,True,False,0.736,False
9,ما هو الناتج الداخلي الخام للمغرب؟,True,False,0.761,False


---
## Section 8 : Interface utilisateur (Gradio)

Interface web RTL en arabe. Le modèle utilisé est configurable via `INTERFACE_MODEL`.


In [20]:
INTERFACE_MODEL = "Qwen3-8B"  # changez ici : "Mistral-7B" ou "Phi-3.5-mini"

def chat_fn(message: str, history: list) -> tuple:
    if not message.strip():
        return "", history
    result = rag_answer(message, INTERFACE_MODEL)
    bot = result['answer']
    if not result['ood'] and result['docs']:
        seen = []
        refs = "\n\n---\n**المصادر:**"
        for doc in result['docs']:
            aid = doc['meta']['article_id']
            if aid not in seen:
                seen.append(aid)
                refs += f"\n- المادة **{aid}** (تطابق: {doc['score']:.2f})"
        bot += refs
    history.append((message, bot))
    return "", history

header_html = (
    '<div style="text-align:center;padding:10px 0">'
    '<h1>🚗 مساعد قانون السير المغربي</h1>'
    '<p style="color:#555">نظام RAG — القانون 52.05 | Powered by HuggingFace API</p>'
    '</div>'
)

info_md = (
    f"**LLM :** `{INTERFACE_MODEL}`\n\n"
    "**Embedding :** `multilingual-e5-small`\n\n"
    f"**Base :** {len(texts)} chunks / {len(df)} articles\n\n"
    f"**FAISS k={RETRIEVAL_K}** | **OOD seuil={OOD_THRESHOLD}**\n\n"
    "---\n⚠️ *Système éducatif — pas une consultation juridique.*"
)

examples_list = [
    "ما هي شروط الحصول على رخصة السياقة في المغرب؟",
    "ما هي عقوبة قيادة السيارة بدون رخصة؟",
    "كم عدد النقاط المخصصة لرخصة السياقة؟",
    "ما هو الحد الأدنى لسن الحصول على رخصة قيادة الدراجة النارية؟",
    "ما هي التزامات السائق المهني في المغرب؟",
]

css = "#header{text-align:center;padding:10px 0 5px}"

with gr.Blocks(title="RAG — قانون السير", theme=gr.themes.Soft(), css=css) as demo:
    gr.HTML(header_html)
    with gr.Row():
        with gr.Column(scale=3):
            chatbot  = gr.Chatbot(label="المحادثة", height=460, rtl=True)
            with gr.Row():
                msg_box  = gr.Textbox(placeholder="اكتب سؤالك هنا…", label="", lines=2, scale=4, rtl=True)
                send_btn = gr.Button("إرسال ▶", variant="primary", scale=1)
            clear_btn = gr.Button("🗑️ مسح", variant="secondary", size="sm")
            gr.Examples(examples=examples_list, inputs=msg_box, label="أمثلة / Exemples")
        with gr.Column(scale=1, min_width=220):
            gr.Markdown(info_md)

    msg_box.submit(chat_fn,  [msg_box, chatbot], [msg_box, chatbot])
    send_btn.click(chat_fn,  [msg_box, chatbot], [msg_box, chatbot])
    clear_btn.click(lambda: ([], ""), outputs=[chatbot, msg_box])
    gr.Markdown("---\n*Devoir 2 NLP — RAG sur textes juridiques arabes (القانون 52.05)*")

print("🚀 Lancement de l'interface Gradio...")
if __import__('os').getenv('LAUNCH_UI', '1') == '1':
    demo.launch(share=True, debug=False, quiet=True)
else:
    print('[UI] Gradio launch skipped (set LAUNCH_UI=1 to enable).')


🚀 Lancement de l'interface Gradio...
* Running on public URL: https://ab1126b0f8cbe94a8e.gradio.live
